# MulViT-TF Fine-tuning
Dual ViT + Transformer Fusion | 2-Phase Transfer Learning

## Hyperparameters

In [ ]:
# ============================================================
# Hyperparameters
# ============================================================
IMG_H, IMG_W   = 240, 320
BATCH_SIZE     = 32

PHASE1_EPOCHS  = 80
PHASE2_EPOCHS  = 120
TOTAL_EPOCHS   = PHASE1_EPOCHS + PHASE2_EPOCHS
LR             = 1e-4
BACKBONE_SCALE = 0.1
WEIGHT_DECAY   = 0.1
DROPOUT        = 0.1
# ============================================================

PRETRAINED_PATH = 'path/to/mulvit_backbone_places365_320x240_best.pth'  # update this path
SAVE_DIR        = 'path/to/save'               # update this path
CKPT_NAME       = 'best.pth'
NUM_LAYERS = 2
DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


## Model

In [ ]:
class ViTEncoder(nn.Module):
    """Compatible with MAE/CLS pretrained weights — ViT backbone (embed_dim=96, depth=6)"""
    def __init__(self, pretrained_path):
        super().__init__()
        self.vit = VisionTransformer(
            img_size=(IMG_H, IMG_W), patch_size=16,
            embed_dim=96, depth=6, num_heads=3,
            mlp_ratio=4., qkv_bias=True, num_classes=0)
        if pretrained_path and os.path.isfile(pretrained_path):
            ckpt  = torch.load(pretrained_path, map_location='cpu', weights_only=False)
            state = ckpt.get('model', ckpt)
            vit_s = {}
            for k, v in state.items():
                if   k.startswith('encoder.'): vit_s[k[8:]] = v
                elif k.startswith('vit.'):     vit_s[k[4:]] = v
                else:                          vit_s[k]     = v
            miss, unex = self.vit.load_state_dict(vit_s, strict=False)
        else:
            pass
        self.vit.pos_embed = nn.Parameter(self.vit.pos_embed.data.clone())

    @property
    def learnable_token(self): return self.vit.cls_token
    @property
    def pos_embed(self):       return self.vit.pos_embed
    @property
    def patch_embed(self):     return self.vit.patch_embed
    @property
    def blocks(self):          return self.vit.blocks
    @property
    def norm(self):            return self.vit.norm

    def forward_features(self, x):
        """Full token sequence [B, 301, 96]  (15x20 patches + CLS)"""
        x = self.vit.patch_embed(x)
        x = self.vit._pos_embed(x)
        if hasattr(self.vit, 'patch_drop'): x = self.vit.patch_drop(x)
        if hasattr(self.vit, 'norm_pre'):   x = self.vit.norm_pre(x)
        x = self.vit.blocks(x)
        x = self.vit.norm(x)
        return x  # [B, 301, 96]

    def forward(self, x): return self.vit(x)  # [B, 96]

class LightTransformerFusion(nn.Module):
    """
    [tokens1 || tokens2] [B,602,96]
    -> seg_embed -> (MHA -> residual -> FFN -> residual) x NUM_LAYERS
    -> CLS1=x[:,0], CLS2=x[:,301] -> concat [B,192]
    """
    def __init__(self, dim=96, num_heads=3, mlp_ratio=2.0, dropout=DROPOUT, num_layers=NUM_LAYERS):
        super().__init__()
        self.seg_embed = nn.Parameter(torch.zeros(1, 2, dim))
        nn.init.trunc_normal_(self.seg_embed, std=0.02)
        self.layers = nn.ModuleList([
            nn.ModuleDict({
                'norm1': nn.LayerNorm(dim),
                'attn':  nn.MultiheadAttention(dim, num_heads, dropout=dropout, batch_first=True),
                'norm2': nn.LayerNorm(dim),
                'ffn':   nn.Sequential(
                    nn.Linear(dim, int(dim * mlp_ratio)), nn.GELU(),
                    nn.Dropout(dropout), nn.Linear(int(dim * mlp_ratio), dim), nn.Dropout(dropout))
            }) for _ in range(num_layers)
        ])
        self.out_norm = nn.LayerNorm(dim * 2)

    def forward(self, tokens1, tokens2):
        t1 = tokens1 + self.seg_embed[:, 0:1]   # [B, 301, 96]
        t2 = tokens2 + self.seg_embed[:, 1:2]
        x  = torch.cat([t1, t2], dim=1)          # [B, 602, 96]
        for layer in self.layers:
            xn = layer['norm1'](x)
            x  = x + layer['attn'](xn, xn, xn)[0]
            x  = x + layer['ffn'](layer['norm2'](x))
        cls1 = x[:, 0]
        cls2 = x[:, 301]
        return self.out_norm(torch.cat([cls1, cls2], dim=-1))  # [B, 192]

class MulViTTF(nn.Module):
    """cam1 + cam2 -> LightTransformerFusion [B,192] -> Linear head -> RSSI"""
    def __init__(self, pretrained_path, dropout=DROPOUT):
        super().__init__()
        self.vit_a  = ViTEncoder(pretrained_path)
        self.vit_b  = ViTEncoder(pretrained_path)
        self.fusion = LightTransformerFusion(dim=96, num_heads=3, mlp_ratio=2.0,
                                             dropout=dropout, num_layers=NUM_LAYERS)
        self.head   = nn.Sequential(
            nn.LayerNorm(192), nn.Dropout(dropout),
            nn.Linear(192, 128), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(128, 1))

    def forward(self, img1, img2):
        t1 = self.vit_a.forward_features(img1)
        t2 = self.vit_b.forward_features(img2)
        return self.head(self.fusion(t1, t2))

    def param_groups(self, lr, backbone_scale):
        bb = set()
        for vn in ['vit_a', 'vit_b']:
            for sub in ['patch_embed', 'blocks', 'norm']:
                bb.add(f'{vn}.vit.{sub}')
        bp, op = [], []
        for name, param in self.named_parameters():
            if '.'.join(name.split('.')[:3]) in bb: bp.append(param)
            else: op.append(param)
        return [{'params':bp,'lr':lr*backbone_scale},{'params':op,'lr':lr}]


## Training

In [ ]:
def set_backbone(model, req):
    for vit in [model.vit_a, model.vit_b]:
        for m in [vit.patch_embed, vit.blocks, vit.norm]:
            for p in m.parameters(): p.requires_grad = req

def phase1_opt(model):
    set_backbone(model, False)
    params = []
    for vit in [model.vit_a, model.vit_b]:
        params += [vit.learnable_token, vit.pos_embed]
    params += list(model.fusion.parameters())
    params += list(model.head.parameters())
    return torch.optim.AdamW(params, lr=LR, weight_decay=WEIGHT_DECAY)

def phase2_opt(model):
    set_backbone(model, True)
    return torch.optim.AdamW(
        model.param_groups(LR, BACKBONE_SCALE), weight_decay=WEIGHT_DECAY)

@torch.no_grad()
def validate(model, loader, criterion):
    model.eval(); total = 0.
    for img1, img2, csi_gt in loader:
        img1, img2, csi_gt = img1.to(DEVICE), img2.to(DEVICE), csi_gt.to(DEVICE)
        with torch.amp.autocast('cuda'):
            total += criterion(model(img1, img2), csi_gt).item()
    return total / len(loader)


model     = MulViTTF(PRETRAINED_PATH).to(DEVICE)
scaler    = GradScaler()
criterion = nn.MSELoss()
best_val  = float('inf')

opt = phase1_opt(model)
sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=PHASE1_EPOCHS, eta_min=LR*0.01)

for epoch in range(1, TOTAL_EPOCHS + 1):
    if epoch == PHASE1_EPOCHS + 1:
        opt = phase2_opt(model)
        sch = torch.optim.lr_scheduler.CosineAnnealingLR(
            opt, T_max=PHASE2_EPOCHS, eta_min=LR*BACKBONE_SCALE*0.01)

    model.train(); tloss = 0.
    for img1, img2, csi_gt in train_loader:
        img1, img2, csi_gt = img1.to(DEVICE), img2.to(DEVICE), csi_gt.to(DEVICE)
        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda'):
            loss = criterion(model(img1, img2), csi_gt)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update()
        tloss += loss.item()

    tloss /= len(train_loader)
    vloss  = validate(model, val_loader, criterion)
    sch.step()

    if vloss < best_val:
        best_val = vloss
        torch.save({'epoch': epoch, 'model': model.state_dict(),
                    'val_loss': vloss}, os.path.join(SAVE_DIR, CKPT_NAME))